# Chapter 17 — Autoencoders and GANs
Companion notebook to `part2_deep_learning/17_autoencoders_gans.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]

## A simple stacked (undercomplete) autoencoder

In [ ]:
stacked_encoder = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(30, activation="selu"),
])
stacked_decoder = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[30]),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28]),
])
stacked_ae = keras.models.Sequential([stacked_encoder, stacked_decoder])
stacked_ae.compile(loss="binary_crossentropy", optimizer="nadam")
history = stacked_ae.fit(X_train, X_train, epochs=10, validation_data=(X_valid, X_valid))

In [ ]:
def plot_reconstructions(model, images, n=5):
    reconstructions = model.predict(images[:n])
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(images[i], cmap="binary"); axes[0, i].axis("off")
        axes[1, i].imshow(reconstructions[i], cmap="binary"); axes[1, i].axis("off")
    axes[0, 0].set_ylabel("Original"); axes[1, 0].set_ylabel("Reconstructed")
    plt.show()

plot_reconstructions(stacked_ae, X_valid)

## A small GAN, trained on Fashion MNIST

In [ ]:
codings_size = 30

generator = keras.models.Sequential([
    keras.layers.Dense(100, activation="selu", input_shape=[codings_size]),
    keras.layers.Dense(150, activation="selu"),
    keras.layers.Dense(28 * 28, activation="sigmoid"),
    keras.layers.Reshape([28, 28]),
])
discriminator = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(150, activation="selu"),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
gan = keras.models.Sequential([generator, discriminator])

discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [ ]:
batch_size = 64
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(1000).batch(batch_size, drop_remainder=True).prefetch(1)

def train_gan(gan, dataset, batch_size, codings_size, n_epochs=5):
    generator, discriminator = gan.layers
    for epoch in range(n_epochs):
        for X_batch in dataset:
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = generator(noise)
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            discriminator.trainable = True
            discriminator.train_on_batch(X_fake_and_real, y1)

            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            discriminator.trainable = False
            gan.train_on_batch(noise, y2)
        print(f"Epoch {epoch + 1}/{n_epochs} done")

train_gan(gan, dataset, batch_size, codings_size, n_epochs=5)

In [ ]:
noise = tf.random.normal(shape=[10, codings_size])
generated = generator(noise)
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i in range(10):
    axes[i].imshow(generated[i], cmap="binary"); axes[i].axis("off")
plt.suptitle("Freshly generated (fake) Fashion-MNIST-like images")
plt.show()

## ✍️ Practice
1. Visualize a few reconstructions above — how much detail is lost at 30 latent dimensions?
2. Why does a denoising autoencoder tend to learn more useful features than a plain one?
3. Train the GAN longer (20-50 epochs) — does image quality visibly improve? Watch for mode collapse (all generated images looking alike).
4. Debug: after training, the generator only produces one type of image regardless of noise input — what's this called, name one mitigation.

In [ ]:
# Your code / notes here